# Planetary Computer imagery and feature-stack workflow

This notebook demonstrates the two optional modules added to this project:

- `planetary_computer.py`: find, preview, and download imagery.
- `feature_stack.py`: composite imagery, add derived features, and create a GeoTIFF that works with the existing classifier.

All cells are intentionally unexecuted. Change the values marked **CHANGE ME** before running them.

## 1. Install the optional dependencies

Run this once in the project environment, then restart the notebook kernel if required.

In [ ]:
# Run in a terminal, not necessarily in this notebook:
# python -m pip install -e ".[planetary-computer]"

# Sentinel-1 RTC may require a configured Planetary Computer account.

## 2. Imports and input placeholders

An AOI can be a GeoJSON/vector file, a GeoJSON geometry, or a WGS84 bounding box `[west, south, east, north]`. Keep the output directory outside version control if downloaded imagery is large.

In [ ]:
from pathlib import Path
import sys
#Allow imports when the notebook is started from notebooks/.
project_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir()
)
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from ML_LC_Classifier import (
    create_feature_stack,
    download_imagery,
    preview_imagery,
    search_imagery,
)

#Area of Interest and output
aoi_path = r"C:\Users\AFahrezi\OneDrive - CIFOR-ICRAF\AFCLIM_TL\AOI_TimorLest\AOI_TL_10km.shp"
#AOI_PATH = project_root / "input_data" / "study_area.geojson"
OUTPUT_DIR = project_root / "input_data" / "planetary_computer"
FEATURE_STACK_PATH = project_root / "input_data" / "feature_stack.tif"

# Alternative: use a WGS84 bounding box instead of AOI_PATH.
# AOI = [106.70, -6.35, 106.95, -6.10]  # [west, south, east, north]

# ===== CHANGE ME: dates, collection, and output grid =====
START_DATE = "2025-01-01"
END_DATE = "2025-12-30"
TARGET_CRS = "EPSG:32751"  
RESOLUTION_METERS = 10
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Sentinel-2 optical imagery

Search first, inspect the results and preview a scene, then download only the bands needed for the feature stack. `B02`, `B03`, `B04`, `B08`, and `B11` support the example indices below.

In [ ]:
sentinel2_scenes = search_imagery(
    aoi=aoi_path,
    collection="sentinel-2-l2a",
    start_date=START_DATE,
    end_date=END_DATE,
    max_cloud_cover=35,  
    max_items=10,
)

for scene in sentinel2_scenes:
    print(scene.item_id, scene.datetime, scene.cloud_cover)
    print("  available assets:", scene.available_assets)

In [ ]:
# Display this URL in a browser or use IPython.display.Image(url=preview_url).
preview_url = preview_imagery(sentinel2_scenes[0], asset="visual")
preview_url

In [ ]:
sentinel2_assets = download_imagery(
    sentinel2_scenes,
    assets=["B02", "B03", "B04", "B08", "B11"],
    output_dir=OUTPUT_DIR / "sentinel2",
)
len(sentinel2_assets)

## 3A. Option 2: stream remote COG windows instead of downloading full scenes

`download_imagery()` above is the easiest workflow, but it downloads whole files. The following optional implementation instead reads only the AOI blocks needed from the signed remote Cloud Optimized GeoTIFFs. Median compositing happens on this computer, one small output window at a time.

Use it for a modest AOI and a limited number of scenes. It does **not** require Dask or an Azure environment. It creates small AOI-only intermediate files, not complete Sentinel-2 tiles.

In [ ]:
# OPTIONAL: imports used only for streamed remote processing.
from contextlib import ExitStack
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window

# This example expects AOI_PATH to be a vector file, set in section 2.
aoi_in_output_crs = gpd.read_file(AOI_PATH).to_crs(TARGET_CRS)
if aoi_in_output_crs.empty:
    raise ValueError("AOI file contains no geometries")

def stream_median_asset(scenes, asset_key, aoi, output_path, *, resolution, block_size=512):
    """Create an AOI-only median composite from remote signed COGs.

    The function opens each signed COG URL with Rasterio. GDAL range requests
    retrieve only blocks that intersect each requested output window, rather
    than downloading whole source scenes.
    """
    west, south, east, north = aoi.total_bounds
    width = int(np.ceil((east - west) / resolution))
    height = int(np.ceil((north - south) / resolution))
    transform = from_bounds(west, south, east, north, width, height)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "float32",
        "crs": aoi.crs,
        "transform": transform,
        "nodata": np.nan,
        "compress": "deflate",
    }

    # search_imagery() already signed each STAC item, so these URLs are ready to read.
    hrefs = [scene.item.assets[asset_key].href for scene in scenes]
    with ExitStack() as stack, rasterio.open(output_path, "w", **profile) as destination:
        sources = [stack.enter_context(rasterio.open(href)) for href in hrefs]
        aligned = [
            stack.enter_context(
                WarpedVRT(
                    source, crs=aoi.crs, transform=transform, width=width, height=height
                )
            )
            for source in sources
        ]

        for row in range(0, height, block_size):
            for col in range(0, width, block_size):
                window = Window(
                    col, row, min(block_size, width - col), min(block_size, height - row)
                )
                values = [
                    np.ma.filled(source.read(1, window=window, masked=True), np.nan)
                    for source in aligned
                ]
                median = np.nanmedian(np.stack(values), axis=0).astype("float32")
                destination.write(median, 1, window=window)
    return output_path

In [ ]:
# OPTIONAL: create AOI-only composites without calling download_imagery().
# CHANGE ME: keep this list small at first. Every asset is read from every selected scene.
STREAMED_ASSETS = ["B02", "B03", "B04", "B08", "B11"]
streamed_dir = OUTPUT_DIR / "streamed_sentinel2"

streamed_sentinel2_assets = [
    stream_median_asset(
        sentinel2_scenes,
        asset_key,
        aoi_in_output_crs,
        streamed_dir / f"{asset_key}.tif",
        resolution=RESOLUTION_METERS,
    )
    for asset_key in STREAMED_ASSETS
]
streamed_sentinel2_assets

To use these streamed composites in section 7, set `all_inputs = streamed_sentinel2_assets`. They are already median composites, so use `composite="first_valid"` in `create_feature_stack()` to avoid an unnecessary second median reduction.

This compact example deliberately does not include pixel-level cloud masking. Add SCL masking before using it for production optical composites; that will be the next improvement when the workflow facade is implemented.

## 4. Optional Sentinel-1 radar imagery

Sentinel-1 RTC commonly provides `vv` and `vh`. The search query below keeps one orbit direction, which is usually preferable when compositing SAR data. Omit this section if radar is not needed.

In [ ]:
# OPTIONAL: run only when radar features are required.
sentinel1_scenes = search_imagery(
    aoi=AOI,
    collection="sentinel-1-rtc",
    start_date=START_DATE,
    end_date=END_DATE,
    query={"sat:orbit_state": {"eq": "ascending"}},  # CHANGE ME if needed
    max_items=10,
)
sentinel1_assets = download_imagery(
    sentinel1_scenes,
    assets=["vv", "vh"],
    output_dir=OUTPUT_DIR / "sentinel1",
)

## 5. Optional Landsat 4--9 imagery

Use `landsat-c2-l2` for Landsat Collection 2 Level-2 scenes. Print `available_assets` first because asset availability can differ by satellite generation. The feature-stack module recognises the common Landsat names `blue`, `green`, `red`, `nir08`, `swir16`, and `swir22` for indices.

In [ ]:
# OPTIONAL: use Landsat instead of, or separately from, Sentinel-2.
landsat_scenes = search_imagery(
    aoi=AOI,
    collection="landsat-c2-l2",
    start_date=START_DATE,
    end_date=END_DATE,
    max_cloud_cover=20,
    max_items=10,
)
print(landsat_scenes[0].available_assets)

# CHANGE ME: choose keys printed above that are common to the selected scenes.
# landsat_assets = download_imagery(landsat_scenes, ["blue", "green", "red", "nir08", "swir16"], OUTPUT_DIR / "landsat")

## 6. Optional STAC DEM and terrain features

This example uses Copernicus DEM. Confirm the height asset name by printing `available_assets`, then map it to the easier feature name `dem`. `terrain_from=["dem"]` adds `dem_slope` and `dem_aspect` to the output.

In [ ]:
# OPTIONAL: run when elevation, slope, and aspect are required.
dem_scenes = search_imagery(
    aoi=AOI,
    collection="cop-dem-glo-30",
    start_date="2020-01-01",  # CHANGE ME only if your chosen DEM requires it
    end_date="2020-12-31",
    max_items=10,
)
print(dem_scenes[0].available_assets)

# CHANGE ME: replace "data" if the printed DEM asset key differs.
# dem_assets = download_imagery(dem_scenes, ["data"], OUTPUT_DIR / "dem")

## 7. Create the final feature stack

Choose the list of inputs required by your project. The following active example uses Sentinel-2 only. Uncomment radar and DEM entries after running their optional sections. The result is a local GeoTIFF and a sidecar JSON manifest.

In [ ]:
all_inputs = [
    *sentinel2_assets,
    # *sentinel1_assets,  # Uncomment after the Sentinel-1 section.
    # *dem_assets,        # Uncomment after the DEM section.
]
# Option 2 alternative: all_inputs = streamed_sentinel2_assets

feature_stack = create_feature_stack(
    imagery=all_inputs,
    output_path=FEATURE_STACK_PATH,
    composite="median",  # Change to "first_valid" when preferred.
    indices=["NDVI", "MNDWI"],
    # asset_aliases={"data": "dem"},
    # terrain_from=["dem"],
    target_crs=TARGET_CRS,
    resolution=RESOLUTION_METERS,
)

print("Feature stack:", feature_stack.stack_path)
print("Manifest:", feature_stack.manifest_path)
print("Band order:", feature_stack.feature_names)

## 8. Use the result in the existing classifier pipeline

The generated GeoTIFF is a normal multiband raster. No existing training or prediction functions need to change. Keep the JSON manifest with the model so the feature order is documented.

In [ ]:
from ML_LC_Classifier import load_and_split_training_data

# ===== CHANGE ME: labelled polygon file and its class column =====
TRAINING_PATH = project_root / "input_data" / "training_samples.geojson"
CLASS_FIELD = "class_id"

X_train, X_test, y_train, y_test = load_and_split_training_data(
    raster_path=feature_stack.stack_path,
    shapefile_path=TRAINING_PATH,
    class_field=CLASS_FIELD,
)

print(X_train.shape, X_test.shape)
# Continue with select_features(), tune_model(), and classify_raster() as in the README.